> Qualitative (GradCam) and Quantative (overlap) analysis for SWIN based model including Auxiliary head, Image only baseline, Cross attention Fusion and Concat Fusion

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

In [17]:
import os
import sys
from decimal import Decimal, ROUND_DOWN

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold
from IPython.display import display

# Run this notebook from the same experiment folder as the old Exp18 notebook.
main_folder = os.path.abspath("../..")
img_folder = os.path.join(main_folder, "data", "train")

assert os.path.isfile(
    os.path.join(main_folder, "src", "gradcam.py")
), "main_folder must point to your project root."

sys.path.insert(0, main_folder)

import src.gradcam as gradcam

print("Using:", gradcam.__file__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BACKBONE = "swin_large_patch4_window12_384"
IMG_SIZE = 384
SWIN_GRID = (12, 12)
FOLDS = [1, 2, 3, 4, 5]

out_extra = os.path.join(
    main_folder,
    "outputs",
    "extra",
    "paper2_corrected_four_model_analysis",
)
os.makedirs(out_extra, exist_ok=True)


def show_four_decimals(frame):
    """Truncate the displayed copy; preserve CSV precision."""
    shown = frame.copy()

    for column in shown.select_dtypes(include=["floating"]).columns:
        shown[column] = shown[column].map(
            lambda value: (
                str(
                    Decimal(str(value)).quantize(
                        Decimal("0.0001"),
                        rounding=ROUND_DOWN,
                    )
                )
                if np.isfinite(value) else str(value)
            )
        )

    display(shown)


print("Device:", device)
print("Output folder:", out_extra)

Using: /home/ghias/Petfinder-Pawpularity-Modular/src/gradcam.py
Device: cuda
Output folder: /home/ghias/Petfinder-Pawpularity-Modular/outputs/extra/paper2_corrected_four_model_analysis


In [18]:
BASELINE_DIR = os.path.join(
    main_folder,
    "outputs",
    "extra",
    "exp3_swin_384_strong_MSE_MLPHead",
)

BCE_DIR = os.path.join(
    main_folder,
    "outputs",
    "extra",
    "exp19_swin_384_strong_MSELoss_All12_Aux_StrongHead_fixDupOcclusion",
)

WBCE_DIR = os.path.join(
    main_folder,
    "outputs",
    "extra",
    "exp18_aux_heads_metadata_weightedBCELoss",
    "exp18_swin_384_MSE_12Aux_weightedBCE_foldTrainWeights",
)

SAL_DIR = os.path.join(
    main_folder,
    "outputs",
    "extra",
    "exp16_swin_384_strong_MSELoss_saliency_strongHead_fixPetBox",
)

AUX_TASKS = [
    "Eyes", "Face", "Occlusion", "Blur",
    "Subject Focus", "Action", "Accessory",
    "Near", "Group", "Collage", "Human", "Info",
]

MODEL_DIRS = {
    "main": BASELINE_DIR,
    "bce12": BCE_DIR,
    "wbce12": WBCE_DIR,
    "saliency": SAL_DIR,
}

model_names = list(MODEL_DIRS)

DISPLAY_NAMES = {
    "main": "Image-only baseline",
    "bce12": "Standard BCE: 12 heads",
    "wbce12": "Corrected weighted BCE: 12 heads",
    "saliency": "Corrected saliency",
}

AUX_MODEL_CONFIG = {
    name: {
        "dir": MODEL_DIRS[name],
        "binary_aux_tasks": AUX_TASKS,
        "head_type": "mlp",
        "head_hidden": 256,
        "is_bce": True,
    }
    for name in ["bce12", "wbce12"]
}

PSEUDO_AUX_CONFIG = {
    "saliency": {
        "dir": SAL_DIR,
        "aux_name": None,
        "use_saliency": False,
        "head_type": "mlp",
    },
}

FUSION_MODEL_CONFIG = {}

for name, directory in MODEL_DIRS.items():
    for fold in FOLDS:
        checkpoint = os.path.join(
            directory, f"model_fold{fold}.pt"
        )
        if not os.path.isfile(checkpoint):
            raise FileNotFoundError(
                f"{name}: missing {checkpoint}"
            )

print("All 20 checkpoints found.")

All 20 checkpoints found.


In [19]:
train_df = pd.read_csv(
    os.path.join(main_folder, "data", "train.csv")
)

boxes_df = pd.read_csv(
    os.path.join(
        main_folder,
        "data",
        "train_brisque_visibility_ratio.csv",
    )
)

box_cols = [
    "yolo_conf",
    "bbox_x1", "bbox_y1",
    "bbox_x2", "bbox_y2",
]

assert train_df["Id"].is_unique
assert boxes_df["Id"].is_unique
assert set(train_df["Id"]).issubset(set(boxes_df["Id"]))

df = train_df.merge(
    boxes_df[["Id"] + box_cols],
    on="Id",
    how="left",
    sort=False,
    validate="one_to_one",
)

assert df["Id"].tolist() == train_df["Id"].tolist()

detected = df["yolo_conf"] > 0.3

assert detected.any(), "No pet detections above confidence 0.3."

assert np.isfinite(
    df.loc[detected, box_cols].to_numpy(dtype=float)
).all(), "Detected images contain invalid box values."

assert (
    df.loc[detected, "bbox_x2"]
    > df.loc[detected, "bbox_x1"]
).all()

assert (
    df.loc[detected, "bbox_y2"]
    > df.loc[detected, "bbox_y1"]
).all()

print("Total images:", len(df))
print("Images with pet detections:", int(detected.sum()))

Total images: 9912
Images with pet detections: 9437


In [20]:
expected_fold = np.zeros(len(df), dtype=np.int64)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (_, val_idx) in enumerate(kf.split(df), start=1):
    expected_fold[val_idx] = fold

rmse_rows = []
fold_rows = []

for name, directory in MODEL_DIRS.items():
    oof = pd.read_csv(
        os.path.join(directory, "oof_detail.csv")
    )

    assert oof["Id"].is_unique, f"{name}: duplicate OOF IDs"

    assert set(oof["Id"]) == set(df["Id"]), (
        f"{name}: OOF image IDs differ from train.csv"
    )

    # Align saved predictions to the original training-image order.
    aligned = oof.set_index("Id").loc[df["Id"]]

    assert np.array_equal(
        aligned["fold"].to_numpy(),
        expected_fold,
    ), (
        f"{name}: saved folds differ from KFold(seed=42). "
        "Resolve this before running Grad-CAM."
    )

    assert np.allclose(
        aligned["ytrue"].to_numpy(),
        df["Pawpularity"].to_numpy(),
        rtol=0,
        atol=1e-6,
    ), f"{name}: saved targets differ"

    residual = (
        aligned["oof_pred"].to_numpy()
        - aligned["ytrue"].to_numpy()
    )

    assert np.isfinite(residual).all()

    fold_rmses = []

    for fold in FOLDS:
        selected = expected_fold == fold

        rmse = float(
            np.sqrt(np.mean(residual[selected] ** 2))
        )

        fold_rmses.append(rmse)

        fold_rows.append({
            "model_name": name,
            "fold": fold,
            "rmse": rmse,
        })

    rmse_rows.append({
        "model_name": name,
        "n_images": len(aligned),
        "pooled_oof_rmse": float(
            np.sqrt(np.mean(residual ** 2))
        ),
        "fold_rmse_mean": float(np.mean(fold_rmses)),
        "fold_rmse_std_population": float(
            np.std(fold_rmses, ddof=0)
        ),
    })

rmse_df = pd.DataFrame(rmse_rows)
fold_rmse_df = pd.DataFrame(fold_rows)

rmse_df.to_csv(
    os.path.join(out_extra, "full_oof_rmse.csv"),
    index=False,
)

fold_rmse_df.to_csv(
    os.path.join(out_extra, "fold_rmse.csv"),
    index=False,
)

print("Fold assignments verified for all four models.")
show_four_decimals(rmse_df)

Fold assignments verified for all four models.


,model_name,n_images,pooled_oof_rmse,fold_rmse_mean,fold_rmse_std_population
0,main,9912,17.7360,17.7336,0.2876
1,bce12,9912,17.7003,17.6983,0.2694
2,wbce12,9912,17.7392,17.7371,0.2713
3,saliency,9912,17.7914,17.7900,0.2231


In [21]:
# Confirm the zero-heatmap correction before the expensive analysis.
assert gradcam.compute_cam_mask_overlap(
    np.zeros((5, 5), dtype=np.float32),
    (1, 1, 3, 3),
    5,
    5,
) == 0.0, "Apply the zero-heatmap fix in src/gradcam.py."

# Confirm that the helper records zero maps.
import inspect

assert "cam_zero" in inspect.getsource(
    gradcam.run_gradcam_quantitative_analysis
), "Add cam_zero recording to src/gradcam.py."

print("Starting four-model, five-fold Grad-CAM analysis...")

img_df, summary_df, corr_df = (
    gradcam.run_gradcam_quantitative_analysis(
        df=df,
        img_folder=img_folder,
        folds=FOLDS,
        model_names=model_names,
        device=device,
        backbone=BACKBONE,
        img_size=IMG_SIZE,
        baseline_dir=BASELINE_DIR,
        aux_model_config=AUX_MODEL_CONFIG,
        fusion_model_config=FUSION_MODEL_CONFIG,
        tab_cols=None,
        grid_size=SWIN_GRID,
        aug_type="strong",
        pseudo_aux_config=PSEUDO_AUX_CONFIG,
        yolo_conf_threshold=0.3,
    )
)

assert not img_df.empty

assert not img_df.duplicated(
    ["fold", "Id", "model_name"]
).any()

assert set(img_df["model_name"]) == set(model_names)

assert img_df.groupby(
    ["fold", "Id"]
)["model_name"].nunique().eq(len(model_names)).all()

assert set(img_df["Id"]) == set(df.loc[detected, "Id"])

assert np.isfinite(
    img_df[
        ["pred_paw", "abs_error",
         "cam_pet_overlap", "cam_face_overlap"]
    ].to_numpy(dtype=float)
).all()

counts_df = (
    img_df.groupby("model_name", as_index=False)
    .agg(
        n_images=("Id", "size"),
        n_zero_cams=("cam_zero", "sum"),
    )
)

summary_df = summary_df.merge(
    counts_df,
    on="model_name",
    validate="one_to_one",
)

img_df.to_csv(
    os.path.join(out_extra, "image_level_results.csv"),
    index=False,
)

summary_df.to_csv(
    os.path.join(out_extra, "overlap_summary.csv"),
    index=False,
)

corr_df.to_csv(
    os.path.join(
        out_extra, "overlap_error_correlations.csv"
    ),
    index=False,
)

print("Overlap summary:")
show_four_decimals(summary_df)

print("Overlap–error correlations:")
show_four_decimals(corr_df)

Starting four-model, five-fold Grad-CAM analysis...


/home/ghias/miniconda3/envs/rapids-env/lib/python3.12/site-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Overlap summary:


,model_name,mean_cam_pet_overlap,mean_cam_face_overlap,n_images,n_zero_cams
0,bce12,0.0622,0.0129,9437,165
1,main,0.0622,0.0135,9437,280
2,saliency,0.0552,0.0124,9437,153
3,wbce12,0.0692,0.0118,9437,375


Overlap–error correlations:


,model_name,pearson_cam_pet_vs_abs_err,spearman_cam_pet_vs_abs_err,pearson_cam_face_vs_abs_err,spearman_cam_face_vs_abs_err
0,bce12,0.1134,0.0516,0.1624,0.1534
1,main,0.1149,0.0403,0.1649,0.1246
2,saliency,0.1195,0.0384,0.1554,0.1209
3,wbce12,0.0948,0.0239,0.1529,0.1172


In [22]:
quartile_df = (
    gradcam.compute_overlap_error_quartile_analysis(
        img_df=img_df,
        model_names=model_names,
        display_names=DISPLAY_NAMES,
        overlap_col="cam_pet_overlap",
        error_col="abs_error",
    )
)

quartile_df.to_csv(
    os.path.join(
        out_extra, "overlap_error_quartiles.csv"
    ),
    index=False,
)

if quartile_df.empty:
    print(
        "No models had distinct low/high "
        "quartile thresholds."
    )
else:
    show_four_decimals(quartile_df)

print("All analysis outputs saved to:")
print(out_extra)

,model_name,display_name,low_overlap_threshold,high_overlap_threshold,low_overlap_mean_overlap,high_overlap_mean_overlap,low_overlap_n,high_overlap_n,low_overlap_mean_error,high_overlap_mean_error,difference,pearson_low_group,spearman_low_group,pearson_high_group,spearman_high_group
0,main,Image-only baseline,0.0023,0.0871,0.0001,0.1817,2360,2360,12.2330,13.9143,1.6812,-0.0687,-0.0905,0.2202,0.1416
1,bce12,Standard BCE: 12 heads,0.0035,0.0867,0.0003,0.1789,2360,2360,11.9441,13.7981,1.8539,-0.0128,-0.0060,0.2175,0.1510
2,wbce12,Corrected weighted BCE: 12 heads,0.0027,0.0983,0.0002,0.1984,2360,2360,12.9426,13.6707,0.7280,-0.0654,-0.0738,0.1997,0.1517
3,saliency,Corrected saliency,0.0016,0.0762,0.0001,0.1625,2360,2360,12.2006,13.7741,1.5735,-0.0353,-0.0458,0.2428,0.1611


All analysis outputs saved to:
/home/ghias/Petfinder-Pawpularity-Modular/outputs/extra/paper2_corrected_four_model_analysis
